In [2]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.read_excel("../datasets/carrot_complete_growth_dataset.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df["Avg_Price"] = (df["Carrot_Min_Price"] + df["Carrot_Max_Price"]) / 2


In [3]:
# Lagged (previous day) price
df["Prev_Price"] = df["Avg_Price"].shift(1)


In [4]:
weather_cols = [
    "Average of temp_avg_75d",
    "Average of humidity_avg_75d",
    "Average of precip_avg_75d",
    "Average of windspeed_avg_75d",
    "Average of solarradiation_avg_75d"
]

# Combine weather into one index
df["Weather_Index"] = df[weather_cols].mean(axis=1)


In [5]:
df["Transport_Index"] = df["Transportation Cost"]


In [6]:
df["Labour_Fert_Index"] = (
    df["Labour Cost"] +
    df["Fertilizer (Urea)"] +
    df["Fertilizer(MOP)"]
)


In [7]:
scaler = MinMaxScaler()

index_cols = [
    "Prev_Price",
    "Weather_Index",
    "Transport_Index",
    "Labour_Fert_Index"
]

df[index_cols] = scaler.fit_transform(df[index_cols])


In [8]:
# =========================
# STEP 7: Formula-based index
# =========================

df["Formula_Price_Index"] = (
    0.55 * df["Prev_Price"] +
    0.25 * df["Weather_Index"] +
    0.15 * df["Transport_Index"] +
    0.05 * df["Labour_Fert_Index"]
)


In [10]:
# Get real price range
price_min = df["Avg_Price"].min()
price_max = df["Avg_Price"].max()

# Convert index to real price
df["Formula_Predicted_Price"] = (
    df["Formula_Price_Index"] * (price_max - price_min) + price_min
)


In [11]:
comparison = df[[
    "Date",
    "Avg_Price",
    "Formula_Predicted_Price"
]].dropna()

comparison.tail(10)


,Date,Avg_Price,Formula_Predicted_Price
1324,2024-12-22,145.0,402.567218
1325,2024-12-23,130.0,411.112443
1326,2024-12-24,110.0,406.731775
1327,2024-12-25,125.0,398.315717
1328,2024-12-26,160.0,412.072862
1329,2024-12-27,225.0,433.362816
1330,2024-12-28,225.0,480.844102
1331,2024-12-29,275.0,487.222841
1332,2024-12-30,220.0,511.707306
1333,2024-12-31,215.0,472.395523


In [12]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# -------------------------
# Load dataset
# -------------------------
df = pd.read_excel("../datasets/carrot_complete_growth_dataset.xlsx")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Base price
df["Avg_Price"] = (df["Carrot_Min_Price"] + df["Carrot_Max_Price"]) / 2

# -------------------------
# Build indices (same as before)
# -------------------------
df["Prev_Price"] = df["Avg_Price"].shift(1)

weather_cols = [
    "Average of temp_avg_75d",
    "Average of humidity_avg_75d",
    "Average of precip_avg_75d",
    "Average of windspeed_avg_75d",
    "Average of solarradiation_avg_75d"
]
df["Weather_Index"] = df[weather_cols].mean(axis=1)

df["Transport_Index"] = df["Transportation Cost"]
df["Labour_Fert_Index"] = (
    df["Labour Cost"] +
    df["Fertilizer (Urea)"] +
    df["Fertilizer(MOP)"]
)

# Scale indices
scaler = MinMaxScaler()
index_cols = ["Prev_Price", "Weather_Index", "Transport_Index", "Labour_Fert_Index"]
df[index_cols] = scaler.fit_transform(df[index_cols])

# -------------------------
# Formula weights
# -------------------------
W_PREV = 0.55
W_WEATHER = 0.25
W_TRANSPORT = 0.15
W_LABOUR = 0.05

# Price range for inverse scaling
price_min = df["Avg_Price"].min()
price_max = df["Avg_Price"].max()

# -------------------------
# Future date range
# -------------------------
future_dates = pd.date_range("2025-01-01", "2025-03-31", freq="D")

# Last known values
last_price = df["Avg_Price"].iloc[-1]
last_weather = df["Weather_Index"].iloc[-1]
last_transport = df["Transport_Index"].iloc[-1]
last_labour = df["Labour_Fert_Index"].iloc[-1]

predictions = []

# -------------------------
# Recursive forecasting
# -------------------------
for date in future_dates:
    prev_scaled = (last_price - price_min) / (price_max - price_min)

    price_index = (
        W_PREV * prev_scaled +
        W_WEATHER * last_weather +
        W_TRANSPORT * last_transport +
        W_LABOUR * last_labour
    )

    predicted_price = price_index * (price_max - price_min) + price_min

    predictions.append({
        "Date": date,
        "Formula_Predicted_Price": predicted_price
    })

    last_price = predicted_price  # feed back

# -------------------------
# Result DataFrame
# -------------------------
forecast_df = pd.DataFrame(predictions)
forecast_df.head()


,Date,Formula_Predicted_Price
0,2025-01-01,469.645523
1,2025-01-02,609.700560
2,2025-01-03,686.730831
3,2025-01-04,729.097480
4,2025-01-05,752.399137


In [13]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# -------------------------
# Load dataset
# -------------------------
df = pd.read_excel("../datasets/carrot_complete_growth_dataset.xlsx")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Base price
df["Avg_Price"] = (df["Carrot_Min_Price"] + df["Carrot_Max_Price"]) / 2

# -------------------------
# Calculate historical spread
# -------------------------
df["Price_Spread"] = df["Carrot_Max_Price"] - df["Carrot_Min_Price"]
avg_spread = df["Price_Spread"].mean()

# -------------------------
# Build indices
# -------------------------
df["Prev_Price"] = df["Avg_Price"].shift(1)

weather_cols = [
    "Average of temp_avg_75d",
    "Average of humidity_avg_75d",
    "Average of precip_avg_75d",
    "Average of windspeed_avg_75d",
    "Average of solarradiation_avg_75d"
]
df["Weather_Index"] = df[weather_cols].mean(axis=1)

df["Transport_Index"] = df["Transportation Cost"]
df["Labour_Fert_Index"] = (
    df["Labour Cost"] +
    df["Fertilizer (Urea)"] +
    df["Fertilizer(MOP)"]
)

# Scale
scaler = MinMaxScaler()
index_cols = ["Prev_Price", "Weather_Index", "Transport_Index", "Labour_Fert_Index"]
df[index_cols] = scaler.fit_transform(df[index_cols])

# -------------------------
# Formula weights
# -------------------------
W_PREV = 0.55
W_WEATHER = 0.25
W_TRANSPORT = 0.15
W_LABOUR = 0.05

price_min = df["Avg_Price"].min()
price_max = df["Avg_Price"].max()

# -------------------------
# Future prediction (Jan–March)
# -------------------------
future_dates = pd.date_range("2025-01-01", "2025-03-31", freq="D")

last_price = df["Avg_Price"].iloc[-1]
last_weather = df["Weather_Index"].iloc[-1]
last_transport = df["Transport_Index"].iloc[-1]
last_labour = df["Labour_Fert_Index"].iloc[-1]

results = []

for date in future_dates:
    prev_scaled = (last_price - price_min) / (price_max - price_min)

    price_index = (
        W_PREV * prev_scaled +
        W_WEATHER * last_weather +
        W_TRANSPORT * last_transport +
        W_LABOUR * last_labour
    )

    avg_price = price_index * (price_max - price_min) + price_min

    min_price = avg_price - (avg_spread / 2)
    max_price = avg_price + (avg_spread / 2)

    results.append({
        "Date": date,
        "Predicted_Min_Price": round(min_price, 2),
        "Predicted_Max_Price": round(max_price, 2),
        "Predicted_Avg_Price": round(avg_price, 2)
    })

    last_price = avg_price

forecast_df = pd.DataFrame(results)
forecast_df.head()


,Date,Predicted_Min_Price,Predicted_Max_Price,Predicted_Avg_Price
0,2025-01-01,445.45,493.84,469.65
1,2025-01-02,585.50,633.90,609.70
2,2025-01-03,662.53,710.93,686.73
3,2025-01-04,704.90,753.29,729.10
4,2025-01-05,728.20,776.60,752.40
